# Droplet Analysis — Google Colab
**DNA nanostar condensate fluorescence microscopy pipeline**

Steps:
1. **Cell 1** — Install packages and download pipeline code from GitHub *(run once per session)*
2. **Cell 2** — Upload your `.czi` file
3. **Cell 3** — Launch the interactive UI: adjust sliders → **▶ Run** → **↺ Re-classify**
4. **Cell 4** — Show analysis plots
5. **Cell 5** — Download results (CSV)

> **Run vs Re-classify:** Use **▶ Run** when changing *detection* parameters or loading a new file. Use **↺ Re-classify** for classification-only changes — skips re-detection and is much faster.

In [ ]:
# Cell 1 — Install packages and download pipeline code (run once per session)
!pip install czifile scikit-image scikit-learn plotly ipywidgets -q

GITHUB_RAW = 'https://raw.githubusercontent.com/YanchengDu/DropletAnalysis/main/pipeline'
!wget -q {GITHUB_RAW}/droplet_pipeline.py -O droplet_pipeline.py
!wget -q {GITHUB_RAW}/droplet_ui.py      -O droplet_ui.py

import importlib, sys
for _mod in ['droplet_pipeline', 'droplet_ui']:
    if _mod in sys.modules:
        importlib.reload(sys.modules[_mod])

from droplet_ui import show_analysis
print('\u2713 Ready.')

In [ ]:
# Cell 2 — Upload your .czi file
from google.colab import files

print('Select your .czi file:')
uploaded = files.upload()

if not uploaded:
    raise RuntimeError('No file uploaded.')

fname    = list(uploaded.keys())[0]
CZI_PATH = f'/tmp/{fname}'
with open(CZI_PATH, 'wb') as f:
    f.write(uploaded[fname])

print(f'\u2713 {fname}  ({len(uploaded[fname])/1e6:.1f} MB) saved to {CZI_PATH}')

In [ ]:
# Cell 3 — Launch interactive UI
# The file path is pre-filled. Adjust sliders then click ▶ Run.
import droplet_ui, importlib; importlib.reload(droplet_ui)
from droplet_ui import launch_ui

S = launch_ui(CZI_PATH)

In [ ]:
# Cell 4 — Show analysis plots (run after pipeline finishes)
show_analysis(
    S['df'],
    S['pixel_size'],
    img_shape=S['norm'].shape if S.get('norm') is not None else None,
)

In [ ]:
# Cell 5 — Download results CSV
import os
from google.colab import files

df_result = S['df']
if df_result is None or len(df_result) == 0:
    print('No results yet — run the pipeline first.')
else:
    df_save = (df_result[df_result['min_confidence'] > 0.05]
               if 'min_confidence' in df_result.columns else df_result).copy()
    base     = os.path.splitext(os.path.basename(CZI_PATH))[0]
    out_path = f'/tmp/{base}_analysis.csv'
    df_save.to_csv(out_path, index=False)
    print(f'Downloading {len(df_save)}/{len(df_result)} confident droplets → {base}_analysis.csv')
    files.download(out_path)